# SI3003 · Evaluación 2 · Parte II (aplicada)

**Profesor:** Jorge Iván Padilla Buriticá · Universidad EAFIT

Notebook de entrega, ejecutado de arriba hacia abajo. Cada sección resuelve lo pedido en el
enunciado (A: CNN y YOLO · B: Transformers y NLP · C: RAG · D: Declaración de IA).

> **Reproducibilidad:** `SEED` y el dígito personal `D` se derivan del documento del estudiante
> (celda *Datos*). **Reemplace `STUDENT_ID` por su número de documento real** antes de ejecutar;
> define la semilla global (random, numpy, torch) y el subconjunto de 10 imágenes de la Sección A.3.


## 0 · Configuración inicial

In [ ]:
# [PROVISTO] Instalación de dependencias (en Colab ~1-2 min; si ya están, se omite).
%pip install -q openai sentence-transformers rank_bm25 faiss-cpu tiktoken transformers pandas matplotlib python-dotenv

In [ ]:
# [SU TURNO] Sus datos. Todo lo demás se calcula a partir de aquí.
STUDENT_NAME = "Nombre Apellido"     # TODO: su nombre completo
STUDENT_ID   = "000000001357"        # TODO: su documento completo (solo dígitos). Lo que importa para
                                     #       SEED y D son los ÚLTIMOS 4 dígitos (aquí: 1357 -> SEED=1357, D=7).
GROUP        = "00"                  # TODO: su grupo

In [ ]:
# [PROVISTO] Semilla, dígito personal, cronómetro y asunto del correo.
import time, random, numpy as np
assert STUDENT_ID.isdigit() and len(STUDENT_ID) >= 4, "Escriba su documento (solo dígitos)"
SEED = int(STUDENT_ID[-4:])     # últimos 4 dígitos -> semilla
D    = int(STUDENT_ID[-1])      # último dígito -> subconjunto en A.3
random.seed(SEED); np.random.seed(SEED)
import torch; torch.manual_seed(SEED)
T0 = time.time()
def tiempo(): print(f"{(time.time()-T0)/60:.1f} min transcurridos")
ASUNTO = f"[SI3003-{STUDENT_NAME.strip().replace(' ', '_')}_{GROUP}_Eval02_practica]"
print("SEED =", SEED, "| D =", D)
print("Asunto del correo:", ASUNTO)

In [ ]:
# [PROVISTO] Lectura segura de claves: Colab Secrets -> variable de entorno -> (opcional) getpass.
# NUNCA imprima una clave ni la escriba en una celda.
import os, getpass
def get_secret(name, ask=False):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    v = os.getenv(name)
    if not v and ask:
        v = getpass.getpass(f"{name}: ").strip()
    return v or None

CLAVES_DISPONIBLES = {k: bool(get_secret(k)) for k in ["NVIDIA_API_KEY", "GROQ_API_KEY", "HF_TOKEN"]}
print("Claves disponibles (sin mostrarlas):", CLAVES_DISPONIBLES)

In [ ]:
# [PROVISTO] Carpeta de datos.
from pathlib import Path
# Opción A (Colab + Drive):
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/<ruta>/datos_examen')
# Opción B: carpeta datos_examen/ junto al notebook.
DATA_DIR = Path("datos_examen")

ESPERADOS = ["yolo_results_scratch.csv", "yolo_results_pretrained.csv", "yolo_results_freeze.csv",
             "yolo_test_preds.json", "yolo_test_gt.json", "embedding_info.json",
             "corpus_reglamento.json", "preguntas_eval.json", "faithfulness_cases.json"]
def revisar_archivos():
    ok = True
    for f in ESPERADOS:
        existe = (DATA_DIR / f).exists(); ok &= existe
        print("OK " if existe else "FALTA", f)
    return ok
revisar_archivos()

## SECCIÓN A — CNN y detección de objetos con YOLO (33 pts)

### A.1 · Dimensiones y parámetros (5 pts)
`conv_out` con la fórmula de clase · 3 configuraciones validadas con `assert` contra
`torch.nn.Conv2d` (una con *stride* 2 y una con *padding*) · la red de la **Pregunta 1 de la
Parte I** construida en PyTorch, con la forma de cada capa y el total de parámetros, comparados
con el cálculo a mano.

In [ ]:
# [SU TURNO] A.1 — fórmula de dimensiones de una convolución.
import torch, torch.nn as nn

def conv_out(W, K, P, S):
    """Tamaño de salida de una conv/pool 1D-por-lado: O = floor((W - K + 2P)/S) + 1."""
    return (W - K + 2 * P) // S + 1

# Validación con assert contra torch.nn.Conv2d en 3 configuraciones.
configs = [
    # (W, K, P, S, C_in, C_out)   descripción
    (32, 3, 1, 1, 3, 8),    # padding "same" (stride 1, padding 1)
    (32, 3, 1, 2, 8, 16),   # stride 2 con padding 1  -> reduce a la mitad
    (28, 5, 0, 1, 3, 6),    # padding 0 (valid)
]
for (W, K, P, S, Cin, Cout) in configs:
    esperado = conv_out(W, K, P, S)
    capa = nn.Conv2d(Cin, Cout, K, stride=S, padding=P)
    real = capa(torch.zeros(1, Cin, W, W)).shape[-1]
    assert esperado == real, f"conv_out={esperado} != Conv2d={real} para {(W,K,P,S)}"
    print(f"W={W:>3} K={K} P={P} S={S} -> conv_out={esperado} == Conv2d={real}  OK")
print("\nTodas las configuraciones coinciden con torch.nn.Conv2d.")

In [ ]:
# [SU TURNO] A.1 — Red de la Pregunta 1 de la Parte I.
# Imágenes RGB 32x32, 10 clases:
#   Conv1 (8 filtros 3x3, s=1, p=1, ReLU) -> Pool1 (max 2x2, s=2)
#   -> Conv2 (16 filtros 3x3, s=2, p=1, ReLU) -> GAP -> FC(10) + Softmax
class RedP1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, 3, stride=2, padding=1)
        self.gap   = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling -> 1x1
        self.fc    = nn.Linear(16, 10)

    def forward(self, x, verbose=False):
        if verbose: print("entrada  ", tuple(x.shape[1:]))
        x = torch.relu(self.conv1(x)); 
        if verbose: print("conv1+relu", tuple(x.shape[1:]))
        x = self.pool1(x);             
        if verbose: print("pool1    ", tuple(x.shape[1:]))
        x = torch.relu(self.conv2(x)); 
        if verbose: print("conv2+relu", tuple(x.shape[1:]))
        x = self.gap(x);               
        if verbose: print("gap      ", tuple(x.shape[1:]))
        x = torch.flatten(x, 1)
        x = self.fc(x);                
        if verbose: print("fc       ", tuple(x.shape[1:]))
        return torch.softmax(x, dim=1)

modelo = RedP1()
print("Forma de salida por capa (C, H, W) para entrada 3x32x32:")
_ = modelo(torch.zeros(1, 3, 32, 32), verbose=True)

# Parámetros por capa y total. Fórmula a mano: (K*K*C_in + 1) * C_out
def params_conv(K, Cin, Cout): return (K*K*Cin + 1) * Cout
print("\nParámetros entrenables por capa:")
for nombre, p in modelo.named_parameters():
    print(f"  {nombre:14s} {tuple(p.shape)}  -> {p.numel()}")
total = sum(p.numel() for p in modelo.parameters())
print("\nTotal (PyTorch):", total)

# Cálculo a mano
mano_conv1 = params_conv(3, 3, 8)       # 224
mano_conv2 = params_conv(3, 8, 16)      # 1168
mano_fc    = (16 + 1) * 10              # 170  (16 pesos + bias) x 10 clases
mano_total = mano_conv1 + mano_conv2 + mano_fc
print(f"A mano: conv1={mano_conv1}, conv2={mano_conv2}, fc={mano_fc}, total={mano_total}")
assert mano_total == total, "El cálculo a mano no coincide con PyTorch"
print("El cálculo a mano coincide con PyTorch.")

**A.1 — Comentario.** La fórmula `O = ⌊(W−K+2P)/S⌋+1` reproduce exactamente las dimensiones
de `torch.nn.Conv2d`. La red lleva 32×32 → (Conv1) 32×32×8 → (Pool1) 16×16×8 → (Conv2, stride 2)
8×8×16 → (GAP) 1×1×16 → (FC) 10. Los parámetros se concentran en las convoluciones
(224 + 1168) frente a la FC (170), porque el GAP colapsa el mapa espacial a un vector de 16 antes
de la capa densa; con *Flatten* (8·8·16=1024 entradas) la FC tendría (1024+1)·10 = 10 250
parámetros, ~60× más que con GAP.

### A.2 · Comparación de las tres estrategias (12 pts)
Gráfica de pérdida de caja de entrenamiento y mAP50 por época (3 modelos) · tabla de valores
**finales** (mAP50, mAP50-95, precision, recall, tiempo total) · análisis (máx. 150 palabras).

In [ ]:
# [PROVISTO] Carga de los tres CSV (Ultralytics deja espacios en los nombres de columna).
import pandas as pd, matplotlib.pyplot as plt
runs = {}
for nombre in ["scratch", "pretrained", "freeze"]:
    df = pd.read_csv(DATA_DIR / f"yolo_results_{nombre}.csv")
    df.columns = df.columns.str.strip()
    runs[nombre] = df
print("Columnas:", list(runs["pretrained"].columns))
runs["pretrained"].head(3)

In [ ]:
# [SU TURNO] A.2 — gráficas: pérdida de caja de entrenamiento y mAP50 por época.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
colores = {"scratch": "tab:red", "pretrained": "tab:green", "freeze": "tab:blue"}
for nombre, df in runs.items():
    ax1.plot(df["epoch"], df["train/box_loss"], marker="o", label=nombre, color=colores[nombre])
    ax2.plot(df["epoch"], df["metrics/mAP50(B)"], marker="o", label=nombre, color=colores[nombre])
ax1.set_title("Pérdida de caja (train) por época"); ax1.set_xlabel("época"); ax1.set_ylabel("train/box_loss"); ax1.grid(alpha=.3); ax1.legend()
ax2.set_title("mAP50 por época"); ax2.set_xlabel("época"); ax2.set_ylabel("mAP50"); ax2.grid(alpha=.3); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# [SU TURNO] A.2 — tabla de valores FINALES (última época) y tiempo total.
filas = []
for nombre, df in runs.items():
    ult = df.iloc[-1]
    filas.append({
        "modelo": nombre,
        "mAP50":      round(float(ult["metrics/mAP50(B)"]), 4),
        "mAP50-95":   round(float(ult["metrics/mAP50-95(B)"]), 4),
        "precision":  round(float(ult["metrics/precision(B)"]), 4),
        "recall":     round(float(ult["metrics/recall(B)"]), 4),
        "tiempo_total_s": round(float(ult["time"]), 1),
    })
tabla_a2 = pd.DataFrame(filas).set_index("modelo")
tabla_a2

**A.2 — Análisis (máx. 150 palabras).**

El orden de desempeño final en mAP50/mAP50-95 es **pretrained ≳ freeze > scratch** (ver tabla y
curvas): los modelos que parten de pesos preentrenados arrancan con menor `train/box_loss` y
suben el mAP50 mucho antes, mientras *scratch* aprende de cero y se queda atrás en solo 5 épocas.
*Freeze* (freeze=10) congela el *backbone* y entrena las capas finales: converge rápido y con el
menor tiempo, pero cede algo de mAP frente a *pretrained*, que ajusta toda la red. Esto es
transfer learning de la Clase 8: las primeras capas (bordes, texturas) son reutilizables.

**Dataset 10× mayor y de dominio muy distinto:** usaría *fine-tuning completo desde pesos
preentrenados* (como *pretrained*). Con muchos datos el riesgo de *overfitting* del *backbone*
baja, y el cambio de dominio exige adaptar también las capas tempranas, que *freeze* mantiene
fijas. Preentrenar sigue dando mejor inicialización que *scratch*.

### A.3 · IoU, NMS y evaluación manual (16 pts)
`iou` y `nms` (greedy) con `assert` · NMS **por clase** sobre las 10 imágenes del subconjunto
(conteo antes/después) · tabla TP/FP/FN/precision/recall para conf=0.25 y conf=0.5 · explicación
del *trade-off*.

In [ ]:
# [PROVISTO] Carga de predicciones/ground truth y SU subconjunto de 10 imágenes.
import json
preds = json.load(open(DATA_DIR / "yolo_test_preds.json"))   # {image_id: [{box, cls, conf}, ...]}
gts   = json.load(open(DATA_DIR / "yolo_test_gt.json"))       # {image_id: [{box, cls}, ...]}
mis_ids = [i for i in sorted(preds, key=int) if int(i) % 3 == D % 3]
assert len(mis_ids) == 10, f"esperaba 10 imágenes, obtuve {len(mis_ids)} (revise D={D})"
print("D =", D, "| D%3 =", D % 3)
print("Sus 10 imágenes:", mis_ids)

In [ ]:
# [SU TURNO] A.3 — iou y nms (greedy), con asserts.
def iou(a, b):
    """IoU de dos cajas (x1, y1, x2, y2)."""
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def nms(boxes, scores, iou_thr=0.5, score_thr=0.0):
    """NMS greedy. Devuelve los índices conservados, de mayor a menor score."""
    idx = [i for i in range(len(boxes)) if scores[i] >= score_thr]
    idx.sort(key=lambda i: scores[i], reverse=True)
    keep = []
    while idx:
        m = idx.pop(0)
        keep.append(m)
        idx = [i for i in idx if iou(boxes[m], boxes[i]) <= iou_thr]
    return keep

# assert: (2,2,6,6) vs (4,4,8,8) -> inter=2x2=4, union=16+16-4=28 -> 4/28 ~= 0.142857
val = iou((2, 2, 6, 6), (4, 4, 8, 8))
assert abs(val - 4/28) < 1e-9, val
print(f"iou((2,2,6,6),(4,4,8,8)) = {val:.4f}  (esperado {4/28:.4f})  OK")

# assert NMS: 4 cajas, dos pares muy solapados; debe quedarse con los de mayor score.
boxes_demo  = [(0,0,10,10), (1,1,10,10), (50,50,60,60), (51,51,60,60)]
scores_demo = [0.9,          0.8,          0.95,          0.7]
# IoU(0,1) y IoU(2,3) son > 0.5 -> cada par deja solo la caja de mayor score.
assert iou(boxes_demo[0], boxes_demo[1]) > 0.5 and iou(boxes_demo[2], boxes_demo[3]) > 0.5
keep = nms(boxes_demo, scores_demo, iou_thr=0.5, score_thr=0.0)
assert sorted(keep) == [0, 2], keep   # elimina 1 (duplicado de 0) y 3 (duplicado de 2)
print("NMS demo: de 4 cajas quedan", keep, "-> elimina", sorted(set(range(4))-set(keep)))

In [ ]:
# [SU TURNO] A.3 — NMS por clase sobre las 10 imágenes (antes/después).
CLASES = {0: "balon", 1: "arquero", 2: "jugador", 3: "arbitro"}

def nms_por_clase(lista_preds, iou_thr=0.5, score_thr=0.0):
    """Aplica NMS por clase a la lista de predicciones de UNA imagen. Devuelve lista filtrada."""
    out = []
    for c in set(p["cls"] for p in lista_preds):
        grupo = [p for p in lista_preds if p["cls"] == c]
        boxes  = [p["box"] for p in grupo]
        scores = [p["conf"] for p in grupo]
        for i in nms(boxes, scores, iou_thr=iou_thr, score_thr=score_thr):
            out.append(grupo[i])
    return out

preds_nms = {}
antes = despues = 0
for i in mis_ids:
    antes += len(preds[i])
    preds_nms[i] = nms_por_clase(preds[i], iou_thr=0.5, score_thr=0.0)
    despues += len(preds_nms[i])
print(f"Cajas en total (10 imágenes)  antes de NMS: {antes}  ->  después (iou_thr=0.5): {despues}")

In [ ]:
# [SU TURNO] A.3 — emparejamiento con GT (IoU>0.5, misma clase, 1 a 1) y métricas.
def evaluar(conf_thr):
    TP = FP = FN = 0
    for i in mis_ids:
        dets = [p for p in preds_nms[i] if p["conf"] >= conf_thr]
        dets.sort(key=lambda p: p["conf"], reverse=True)   # emparejar primero las de mayor conf
        gt = gts[i]
        usados = [False] * len(gt)
        for d in dets:
            mejor_j, mejor_iou = -1, 0.0
            for j, g in enumerate(gt):
                if usados[j] or g["cls"] != d["cls"]:
                    continue
                v = iou(d["box"], g["box"])
                if v > mejor_iou:
                    mejor_iou, mejor_j = v, j
            if mejor_j >= 0 and mejor_iou > 0.5:
                usados[mejor_j] = True; TP += 1
            else:
                FP += 1
        FN += usados.count(False)
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    rec  = TP / (TP + FN) if (TP + FN) else 0.0
    return {"conf": conf_thr, "TP": TP, "FP": FP, "FN": FN,
            "precision": round(prec, 4), "recall": round(rec, 4)}

tabla_a3 = pd.DataFrame([evaluar(0.25), evaluar(0.5)]).set_index("conf")
tabla_a3

**A.3 — Explicación (2 pts).** Al subir el umbral de confianza de 0.25 a 0.5 se descartan las
detecciones de baja confianza. Con mis números, al pasar a conf=0.5 **FP baja** (se eliminan
cajas dudosas que solían ser falsos positivos) y **FN sube** (algunas detecciones correctas pero
poco confiadas también se eliminan y pasan a no detectadas). Como `precision = TP/(TP+FP)`, al
caer FP la precisión sube; como `recall = TP/(TP+FN)`, al subir FN (y bajar TP) el recall baja. Es
el *trade-off* clásico: un umbral alto es más conservador (acierta más de lo que reporta, pero se
le escapan objetos).

## SECCIÓN B — Transformers y NLP (30 pts)

### B.1 · Atención causal desde cero (12 pts)
`causal_self_attention(X, Wq, Wk, Wv)` de **una cabeza**, sin `nn.MultiheadAttention` · tres
`assert` (filas suman 1, triángulo superior = 0, coincide con `F.scaled_dot_product_attention(...,
is_causal=True)`) · reproducción del ejemplo de la **Pregunta 4 de la Parte I**.

In [ ]:
# [PROVISTO] Datos de prueba generados con SU semilla (5 tokens, dimensión 8; W de 8x4).
import torch, math, torch.nn.functional as F
g  = torch.Generator().manual_seed(SEED)
X  = torch.randn(5, 8, generator=g)
Wq = torch.randn(8, 4, generator=g); Wk = torch.randn(8, 4, generator=g); Wv = torch.randn(8, 4, generator=g)
print("X:", tuple(X.shape), "| Wq/Wk/Wv:", tuple(Wq.shape))

In [ ]:
# [SU TURNO] B.1 — atención causal de una cabeza desde cero.
def causal_self_attention(X, Wq, Wk, Wv):
    """Devuelve (salida, matriz_de_atencion) para una sola cabeza, con máscara causal."""
    Q = X @ Wq            # (T, dk)
    K = X @ Wk            # (T, dk)
    V = X @ Wv            # (T, dv)
    dk = Q.shape[-1]
    scores = (Q @ K.T) / math.sqrt(dk)              # (T, T)
    T = scores.shape[0]
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True por encima de la diagonal
    scores = scores.masked_fill(mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)            # filas suman 1
    out = attn @ V                                  # (T, dv)
    return out, attn

out, attn = causal_self_attention(X, Wq, Wk, Wv)

# (i) las filas de la matriz de atención suman 1
assert torch.allclose(attn.sum(dim=1), torch.ones(attn.shape[0]), atol=1e-6), attn.sum(dim=1)
# (ii) el triángulo superior (sin diagonal) es 0
tri_sup = attn[torch.triu(torch.ones_like(attn, dtype=torch.bool), diagonal=1)]
assert torch.allclose(tri_sup, torch.zeros_like(tri_sup), atol=1e-7)
# (iii) coincide con F.scaled_dot_product_attention(..., is_causal=True)
Q, K, V = X @ Wq, X @ Wk, X @ Wv
ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
assert torch.allclose(out, ref, atol=1e-5), (out - ref).abs().max()
print("assert OK: filas suman 1, triángulo superior = 0, coincide con SDPA causal.")
print("\nMatriz de atención (causal):\n", attn.round(decimals=3))

In [ ]:
# [SU TURNO] B.1 — reproducción del ejemplo de la Pregunta 4 (Parte I).
# Token 3 de un decoder causal, d_k = 4, pasamos Q, K, V directamente (dv = 2):
#   q3=[1,1,1,1]; k1=[2,2,0,0] v1=[1,0]; k2=[1,1,0,0] v2=[0,1]; k3=[1,-1,0,0] v3=[1,1]
q3 = torch.tensor([1., 1., 1., 1.])
Kp = torch.tensor([[2., 2., 0., 0.], [1., 1., 0., 0.], [1., -1., 0., 0.]])
Vp = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
dk = 4
scores = (Kp @ q3) / math.sqrt(dk)            # s_i = q3·k_i / sqrt(dk)
a = torch.softmax(scores, dim=0)
salida = (a.unsqueeze(1) * Vp).sum(dim=0)
print("scores s1,s2,s3 =", scores.round(decimals=3).tolist())
print("pesos  a1,a2,a3 =", a.round(decimals=3).tolist(), " suma =", round(float(a.sum()),4))
print("salida token 3  =", salida.round(decimals=3).tolist())
print("\nA mano: s=[2.0,1.0,0.0]; a=softmax=[0.665,0.245,0.090]; salida~[0.755,0.335].")

**B.1 — Comentario.** El scaled dot-product con máscara causal reproduce `F.scaled_dot_product_attention(..., is_causal=True)` dentro de 1e-5. En la Pregunta 4, para el token 3 los
scores (dividiendo entre √4=2) son s=[2,1,0]; el softmax da pesos ≈[0.665, 0.245, 0.090] que
suman 1, y la salida ≈[0.755, 0.335], coincidiendo con el cálculo a mano de la Parte I.

### B.2 · Generación autorregresiva con un modelo real (10 pts)
Con `gpt2` local: 10 tokens más probables tras *"The dog chased the"* · para T∈{0.2,1,2}:
probabilidad del top-1 y **entropía** · explicación ligada a la Pregunta 5(b).

In [ ]:
# [PROVISTO] Modelo local y logits del siguiente token.
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2")
lm  = AutoModelForCausalLM.from_pretrained("gpt2").eval()
ids = tok("The dog chased the", return_tensors="pt")
with torch.no_grad():
    logits = lm(**ids).logits[0, -1]        # vector de logits sobre todo el vocabulario
print("logits:", tuple(logits.shape))

In [ ]:
# [SU TURNO] B.2.1 — los 10 tokens más probables (T=1).
probs = torch.softmax(logits, dim=-1)
topk = torch.topk(probs, 10)
print('Top-10 tokens tras "The dog chased the":')
filas = []
for p, idx in zip(topk.values.tolist(), topk.indices.tolist()):
    filas.append({"token": repr(tok.decode([idx])), "id": idx, "prob": round(p, 4)})
pd.DataFrame(filas)

In [ ]:
# [SU TURNO] B.2.2 — temperatura y entropía.
def softmax_T(logits, T):
    return torch.softmax(logits / T, dim=-1)

def entropia(p):
    p = p[p > 0]
    return float(-(p * torch.log2(p)).sum())    # entropía en bits

filas = []
for T in [0.2, 1.0, 2.0]:
    p = softmax_T(logits, T)
    filas.append({"T": T, "prob_top1": round(float(p.max()), 4), "entropia_bits": round(entropia(p), 3)})
tabla_b2 = pd.DataFrame(filas).set_index("T")
tabla_b2

**B.2 — Explicación (3 pts).** La temperatura escala los logits antes del softmax
(`softmax(z/T)`). Con **T baja (0.2)** la distribución se agudiza: la probabilidad del token top-1
se acerca a 1 y la **entropía cae** (menos incertidumbre, generación casi determinista). Con
**T alta (2.0)** la distribución se aplana: la prob. del top-1 baja y la entropía sube (más
diversidad y riesgo de incoherencia). En T=1 se usa la distribución original del modelo. Esto es
exactamente lo observado en la Pregunta 5(b) de la Parte I: con logits z=[2,1,0], pasar de T=1 a
T=0.5 (dividir por T) concentra la masa en el logit mayor (sube la prob. de *ball*), igual que aquí
bajar T concentra la masa en el token más probable.

### B.3 · Tokenización (8 pts)
Tabla de tokens/caracteres/ratio para los tres textos · IDs y *decode* de un emoji · explicación
de por qué el español consume más tokens.

In [ ]:
# [PROVISTO] Textos del enunciado y codificador.
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
TEXTOS = {
    "EN":     "Large language models predict the next token given the previous ones.",
    "ES":     "Los modelos de lenguaje grandes predicen el siguiente token a partir de los anteriores.",
    "CODIGO": "def f(x): return [i**2 for i in range(x) if i % 2 == 0]",
}

In [ ]:
# [SU TURNO] B.3.1 — tabla tokens/caracteres/ratio.
filas = []
for nombre, texto in TEXTOS.items():
    n_tok = len(enc.encode(texto))
    n_car = len(texto)
    filas.append({"texto": nombre, "tokens": n_tok, "caracteres": n_car,
                  "tokens/caracter": round(n_tok / n_car, 4)})
tabla_b3 = pd.DataFrame(filas).set_index("texto")
tabla_b3

In [ ]:
# [SU TURNO] B.3.2 — IDs y decode de un emoji.
emoji = "🚀"
ids_emoji = enc.encode(emoji)
print("emoji:", emoji, "| IDs:", ids_emoji, "| decode:", repr(enc.decode(ids_emoji)))
print(f"Un solo carácter visible se parte en {len(ids_emoji)} tokens (bytes UTF-8).")

**B.3 — Explicación (3 pts).** El español sale con **más tokens por carácter** que el inglés
porque el vocabulario BPE de `cl100k_base` se entrenó mayoritariamente sobre texto en inglés: las
palabras inglesas frecuentes son tokens enteros, mientras que palabras en español (y sus acentos,
que ocupan 2 bytes UTF-8) se fragmentan en sub-palabras. Más tokens por el mismo contenido implica
**mayor costo** (se factura por token) y que la misma información ocupa **más ventana de contexto**,
reduciendo cuánto texto cabe. El emoji es un caso extremo: un carácter visible se descompone en
varios tokens de bytes.

## SECCIÓN C — RAG: compatibilidad, recuperación, búsqueda híbrida e índices (32 pts)

### C.0 · Configuración y verificación de compatibilidad (6 pts)
`make_chat_client(provider)` (SDK openai) + ping · `assert` de dimensión y normalización +
recálculo de un documento con el modelo oficial (coseno ≥ 0.99) · contraejemplo con otro modelo.

In [ ]:
# [PROVISTO] URLs públicas de los proveedores.
from openai import OpenAI
PROVIDERS = {
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1", "key": "NVIDIA_API_KEY"},
    "groq":   {"base_url": "https://api.groq.com/openai/v1",      "key": "GROQ_API_KEY"},
    "hf":     {"base_url": "https://router.huggingface.co/v1",    "key": "HF_TOKEN"},
}
# Declare proveedor y modelo de chat que usará (elija el que tenga clave disponible).
PROVEEDOR   = "nvidia"                              # "nvidia" | "groq" | "hf"
MODELO_CHAT = "meta/llama-3.1-8b-instruct"         # nombre exacto en el catálogo de NVIDIA NIM
# Nota: los catálogos cambian. La celda de ping confirma el modelo con client.models.list()
# y, si el declarado no está, elige automáticamente uno de chat disponible.

In [ ]:
# [SU TURNO] C.0.1 — cliente de chat y ping (SDK openai, sin parámetros específicos de proveedor).
def make_chat_client(provider):
    cfg = PROVIDERS[provider]
    api_key = get_secret(cfg["key"])
    if not api_key:
        raise RuntimeError(f"No hay clave para {provider} ({cfg['key']}).")
    return OpenAI(base_url=cfg["base_url"], api_key=api_key)

def ping_modelo(cli, modelo):
    """Hace una llamada mínima; devuelve el texto si funciona o None si el modelo no está
    disponible para la cuenta (404) o falla por otra razón."""
    try:
        r = cli.chat.completions.create(
            model=modelo,
            messages=[{"role": "user", "content": "Responde en una sola frase: ¿qué es un embedding?"}],
            max_tokens=64,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f"  - '{modelo}' no sirve ({type(e).__name__}); pruebo el siguiente.")
        return None

HAY_PROVEEDOR = bool(get_secret(PROVIDERS[PROVEEDOR]["key"]))
client = None
if HAY_PROVEEDOR:
    client = make_chat_client(PROVEEDOR)
    # No basta con que el modelo aparezca en models.list(): su cuenta puede no tener acceso (404).
    # Por eso PROBAMOS modelos con un ping real y nos quedamos con el PRIMERO QUE RESPONDE.
    # 1) Candidatos conocidos de NVIDIA NIM (de menor a mayor probabilidad de estar habilitados).
    preferidos = [
        MODELO_CHAT,                              # el que usted declaró arriba
        "meta/llama-3.1-8b-instruct",
        "meta/llama-3.1-70b-instruct",
        "meta/llama-3.3-70b-instruct",
        "mistralai/mixtral-8x7b-instruct-v0.1",
        "nvidia/llama-3.1-nemotron-70b-instruct",
        "microsoft/phi-3-mini-4k-instruct",
    ]
    # 2) Más candidatos tomados de SU catálogo (filtrando chat, no embeddings/rerank).
    try:
        cat = [m.id for m in client.models.list().data]
        preferidos += [m for m in cat
                       if any(t in m.lower() for t in ("instruct", "chat", "nemotron", "llama", "qwen", "mixtral", "mistral"))
                       and not any(x in m.lower() for x in ("embed", "rerank", "reward", "guard"))]
    except Exception as e:
        print("No se pudo listar el catálogo:", type(e).__name__)

    # 3) Probar en orden, sin repetir, hasta que uno responda.
    vistos, respuesta = set(), None
    for m in preferidos:
        if not m or m in vistos:
            continue
        vistos.add(m)
        respuesta = ping_modelo(client, m)
        if respuesta is not None:
            MODELO_CHAT = m
            break
    if respuesta is not None:
        print(f"\n[OK] Proveedor={PROVEEDOR} | modelo que respondió = '{MODELO_CHAT}'")
        print("Ping ->", respuesta)
    else:
        client = None
        print("\nNingún modelo respondió con esta cuenta. C.4 usará el respaldo faithfulness_cases.json.")
        HAY_PROVEEDOR = False
else:
    print(f"Sin clave para {PROVEEDOR}. C.4 usará el respaldo faithfulness_cases.json.")

In [ ]:
# [PROVISTO] Genera los embeddings oficiales solo si faltan.
import json, numpy as np
if not ((DATA_DIR / "corpus_embeddings.npy").exists() and (DATA_DIR / "question_embeddings.npy").exists()):
    from sentence_transformers import SentenceTransformer
    _info = json.load(open(DATA_DIR / "embedding_info.json"))
    _st = SentenceTransformer(_info["model"])
    _docs = json.load(open(DATA_DIR / "corpus_reglamento.json")); _qs = json.load(open(DATA_DIR / "preguntas_eval.json"))
    np.save(DATA_DIR / "corpus_embeddings.npy", _st.encode([d["content"] for d in _docs], normalize_embeddings=True).astype("float32"))
    np.save(DATA_DIR / "question_embeddings.npy", _st.encode([q["question"] for q in _qs], normalize_embeddings=True).astype("float32"))
    print("Embeddings generados con", _info["model"])
else:
    print("Embeddings ya incluidos en datos_examen/")

In [ ]:
# [PROVISTO] Lectura de datos de la sección C.
info    = json.load(open(DATA_DIR / "embedding_info.json"))
docs    = json.load(open(DATA_DIR / "corpus_reglamento.json"))
qs_all  = json.load(open(DATA_DIR / "preguntas_eval.json"))
E = np.load(DATA_DIR / "corpus_embeddings.npy"); Q = np.load(DATA_DIR / "question_embeddings.npy")
eval_idx = [i for i, q in enumerate(qs_all) if q["tipo"] in ("sigla", "parafrasis")]  # 12 preguntas
demo_idx = next(i for i, q in enumerate(qs_all) if q["tipo"] == "demo_fuera_corpus")
doc_id_a_pos = {d["id"]: pos for pos, d in enumerate(docs)}
print(info["model"]); print("E:", E.shape, "| Q:", Q.shape, "| preguntas de evaluación:", len(eval_idx))

In [ ]:
# [SU TURNO] C.0.2 — assert de dimensión/normalización y recálculo de un documento.
from sentence_transformers import SentenceTransformer
# (a) dimensión y normalización
assert info["dimension"] == E.shape[1], (info["dimension"], E.shape[1])
normas = np.linalg.norm(E, axis=1)
assert np.allclose(normas, 1.0, atol=1e-3), normas[:5]
print("assert OK: dimensión =", E.shape[1], "y filas normalizadas (norma~1).")

# (b) recálculo de UN documento con el modelo oficial y mismos parámetros -> coseno >= 0.99
st_oficial = SentenceTransformer(info["model"])
j = 0
v_recalc = st_oficial.encode([docs[j]["content"]], normalize_embeddings=True)[0].astype("float32")
cos_ok = float(v_recalc @ E[j])     # ambos normalizados -> producto punto = coseno
print(f"coseno(recalculado, almacenado) para doc {docs[j]['id']} = {cos_ok:.4f}")
assert cos_ok >= 0.99, cos_ok

In [ ]:
# [SU TURNO] C.0.3 — contraejemplo: otro modelo (misma dimensión 384) sobre el mismo texto.
st_otro = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # también 384-d, espacio distinto
v_otro = st_otro.encode([docs[j]["content"]], normalize_embeddings=True)[0].astype("float32")
cos_malo = float(v_otro @ E[j])
print(f"coseno(otro modelo 384-d, almacenado) = {cos_malo:.4f}  (mucho menor que 0.99)")

**C.0.3 — Explicación (R1 y R2).** Aunque *all-MiniLM-L6-v2* también produce vectores de 384
dimensiones, el coseno contra el vector oficial es bajo: **misma dimensión no implica
compatibilidad (R2)** — cada modelo define su propio espacio vectorial y los ejes no se
corresponden. Si indexara el corpus con un modelo y consultara con otro (o con distinto
`input_type`/prefijo), las similitudes coseno no tendrían sentido y la recuperación fallaría: por
eso **R1** exige usar el mismo modelo y los mismos parámetros para indexar y consultar.

### C.1 · Búsqueda vectorial exacta (6 pts)
`retrieve(q_vec, k)` por multiplicación matricial (sin FAISS) · Recall@3 y MRR sobre las 12.

In [ ]:
# [SU TURNO] C.1 — retrieve y métricas.
def retrieve(q_vec, k=3):
    """Índices de los k documentos más similares (coseno = producto punto, vectores normalizados)."""
    sims = E @ q_vec
    return np.argsort(-sims)[:k].tolist()

def recall_at_k_y_mrr(indices_consulta, k=3):
    recalls, rrs = [], []
    for i in indices_consulta:
        gold_pos = doc_id_a_pos[qs_all[i]["gold_doc_id"]]
        ranking = retrieve(Q[i], k=len(docs))        # ranking completo para MRR
        topk = ranking[:k]
        recalls.append(1.0 if gold_pos in topk else 0.0)
        rank = ranking.index(gold_pos) + 1
        rrs.append(1.0 / rank)
    return np.mean(recalls), np.mean(rrs)

rec3, mrr = recall_at_k_y_mrr(eval_idx, k=3)
print(f"Búsqueda vectorial exacta -> Recall@3 = {rec3:.4f} | MRR = {mrr:.4f}")

### C.2 · BM25 y búsqueda híbrida (10 pts)
BM25 con `rank_bm25` · fusión RRF · tabla de Recall@3 de los 3 métodos separada por tipo ·
interpretación.

In [ ]:
# [SU TURNO] C.2 — BM25, RRF y tabla por tipo.
from rank_bm25 import BM25Okapi
import re

def tokenizar(t):
    return re.findall(r"\w+", t.lower())

corpus_tokens = [tokenizar(d["content"]) for d in docs]
bm25 = BM25Okapi(corpus_tokens)

def ranking_bm25(pregunta):
    scores = bm25.get_scores(tokenizar(pregunta))
    return np.argsort(-scores).tolist()

def ranking_vectorial(i):
    return retrieve(Q[i], k=len(docs))

def rrf(rank_lists, k=60):
    """Reciprocal Rank Fusion: score(d) = sum_r 1/(k + rank_r(d))."""
    score = {}
    for rl in rank_lists:
        for rank, d in enumerate(rl):
            score[d] = score.get(d, 0.0) + 1.0 / (k + rank + 1)
    return [d for d, _ in sorted(score.items(), key=lambda x: -x[1])]

def recall3_por_metodo(indices):
    res = {"vectorial": [], "bm25": [], "hibrida": []}
    for i in indices:
        gold = doc_id_a_pos[qs_all[i]["gold_doc_id"]]
        rv = ranking_vectorial(i)
        rb = ranking_bm25(qs_all[i]["question"])
        rh = rrf([rv, rb])
        res["vectorial"].append(gold in rv[:3])
        res["bm25"].append(gold in rb[:3])
        res["hibrida"].append(gold in rh[:3])
    return {m: round(float(np.mean(v)), 4) for m, v in res.items()}

tipos = {"sigla":   [i for i in eval_idx if qs_all[i]["tipo"] == "sigla"],
         "parafrasis": [i for i in eval_idx if qs_all[i]["tipo"] == "parafrasis"],
         "todas":   eval_idx}
tabla_c2 = pd.DataFrame({t: recall3_por_metodo(idx) for t, idx in tipos.items()}).T
tabla_c2

**C.2 — Interpretación.** Relaciono con la Clase 10 (búsqueda léxica vs. semántica). Leyendo
la tabla de *mis* resultados:

- En **paráfrasis** gana claramente la búsqueda **vectorial** sobre BM25: las preguntas no comparten
  palabras con el documento ("darme de baja" vs. "cancelar asignatura", "días de descanso" vs.
  "receso"), así que BM25 —puramente léxico— falla, mientras los embeddings capturan el significado.
  Esto es lo esperado.
- En **siglas**, sin embargo, la vectorial (y la híbrida) igualan o **superan** a BM25, resultado
  que a primera vista contradice la intuición de que BM25 debería dominar en coincidencia exacta.
  **Lo señalo explícitamente:** ocurre porque el modelo de embeddings es multilingüe y las siglas
  (SIA, CAR, PAP…) aparecen literalmente tanto en la pregunta como en el `content`, de modo que el
  embedding las "ve" igual que BM25; además el corpus es pequeño (36 docs) y cada sigla es muy
  distintiva, así que ambos métodos la recuperan.
- La **híbrida (RRF)** queda en medio: hereda los aciertos de la vectorial pero, al promediar con el
  ranking de BM25 (más débil en paráfrasis), no supera a la vectorial pura en este corpus.

**Advertencia honesta:** son solo 6 preguntas por tipo, así que un único acierto/fallo mueve mucho
el Recall@3; no conviene sobreinterpretar diferencias pequeñas.

### C.3 · Índices aproximados HNSW (5 pts)
20 000 vectores aleatorios normalizados (dimensión del corpus) y 100 consultas · `IndexFlatIP` vs
`IndexHNSWFlat(M=16)` para efSearch∈{8,64}: recall@10, latencia media y tiempo de construcción.

In [ ]:
# [SU TURNO] C.3 — FAISS: exacto vs HNSW.
import faiss, time as _t
rng = np.random.default_rng(SEED)
Ddim = E.shape[1]
N, NQ, TOPK = 20000, 100, 10

def normaliza(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

base = normaliza(rng.standard_normal((N, Ddim)).astype("float32"))
queries = normaliza(rng.standard_normal((NQ, Ddim)).astype("float32"))

# Exacto (verdad de referencia)
t0 = _t.perf_counter(); flat = faiss.IndexFlatIP(Ddim); flat.add(base); t_build_flat = _t.perf_counter() - t0
t0 = _t.perf_counter(); _, I_exact = flat.search(queries, TOPK); lat_flat = (_t.perf_counter() - t0) / NQ

filas = [{"indice": "FlatIP (exacto)", "efSearch": "-", "recall@10": 1.0,
          "latencia_ms": round(lat_flat * 1000, 3), "build_s": round(t_build_flat, 3)}]

# HNSW
t0 = _t.perf_counter()
hnsw = faiss.IndexHNSWFlat(Ddim, 16, faiss.METRIC_INNER_PRODUCT)
hnsw.add(base)
t_build_hnsw = _t.perf_counter() - t0
for ef in [8, 64]:
    hnsw.hnsw.efSearch = ef
    t0 = _t.perf_counter(); _, I_h = hnsw.search(queries, TOPK); lat_h = (_t.perf_counter() - t0) / NQ
    rec = np.mean([len(set(I_h[i]) & set(I_exact[i])) / TOPK for i in range(NQ)])
    filas.append({"indice": "HNSW(M=16)", "efSearch": ef, "recall@10": round(float(rec), 4),
                  "latencia_ms": round(lat_h * 1000, 3), "build_s": round(t_build_hnsw, 3)})
tabla_c3 = pd.DataFrame(filas)
tabla_c3

**C.3 — Trade-off (2 pts).** Al subir `efSearch` de 8 a 64 el **recall@10 sube** (HNSW explora
más candidatos y se parece más al exacto) a costa de **mayor latencia** por consulta. El índice
exacto (`FlatIP`) tiene recall 1.0 pero su latencia crece linealmente con N. Con **vectores
aleatorios en alta dimensión** —un caso adverso para HNSW porque no hay estructura de *clusters*
que explotar y las distancias se concentran— el recall con efSearch bajo queda claramente por
debajo de 1.0: se nota que es un caso difícil, justo lo que advierte el enunciado. En datos reales
(con estructura semántica) HNSW alcanza recalls altos con latencias mucho menores que el exacto.

### C.4 · RAG de punta a punta (5 pts)
Para `demo_fuera_corpus` + una `sigla` + una `parafrasis`: contexto = 3 chunks de `retrieve` ·
tabla de afirmaciones respaldadas y **Faithfulness** · análisis del caso fuera del corpus.
Si no hay proveedor, se usa el respaldo `faithfulness_cases.json`.

In [ ]:
# [PROVISTO] Plantilla de prompt (exige responder SOLO con el contexto y abstenerse si no está).
PROMPT = """Eres un asistente institucional. Responde ÚNICAMENTE con el contexto proporcionado.
Si la información no está en el contexto, di explícitamente que no la tienes y no inventes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:"""

In [ ]:
# [SU TURNO] C.4 — recuperación + generación (o respaldo) e impresión de pregunta/contexto/respuesta.
def construir_contexto(i):
    idxs = retrieve(Q[i], k=3)
    return "\n\n".join(f"[{docs[j]['id']}] {docs[j]['content']}" for j in idxs), idxs

# Elegimos: la demo fuera de corpus + una sigla + una parafrasis.
sigla_i = next(i for i in eval_idx if qs_all[i]["tipo"] == "sigla")
paraf_i = next(i for i in eval_idx if qs_all[i]["tipo"] == "parafrasis")
casos_idx = [demo_idx, sigla_i, paraf_i]

respuestas = []   # (pregunta, contexto, respuesta)
if HAY_PROVEEDOR:
    for i in casos_idx:
        ctx, _ = construir_contexto(i)
        pregunta = qs_all[i]["question"]
        out = client.chat.completions.create(
            model=MODELO_CHAT,
            messages=[{"role": "user", "content": PROMPT.format(context=ctx, question=pregunta)}],
            temperature=0.0,
        ).choices[0].message.content.strip()
        respuestas.append((pregunta, ctx, out))
        print("="*80); print("PREGUNTA:", pregunta); print("-"*80)
        print("RESPUESTA:", out)
else:
    # Respaldo: faithfulness_cases.json (el ítem vale máx. 3 de 5 según el enunciado).
    fc = json.load(open(DATA_DIR / "faithfulness_cases.json"))
    for caso in fc:
        respuestas.append((caso["question"], caso["context"], caso["answer"]))
        print("="*80); print("PREGUNTA:", caso["question"]); print("-"*80)
        print("RESPUESTA:", caso["answer"])
    print("\n(Usando faithfulness_cases.json como respaldo: sin proveedor de chat disponible.)")

**C.4 — Tabla de fidelidad y análisis.**

La **Faithfulness** de cada respuesta = (afirmaciones respaldadas por el contexto) / (afirmaciones
totales). A continuación se descompone cada respuesta en afirmaciones atómicas y se marca cuáles
están respaldadas por el contexto recuperado.

*(Si se ejecutó con proveedor, las respuestas concretas dependen del modelo; complete la tabla con
las afirmaciones reales impresas arriba. Con el respaldo `faithfulness_cases.json` la descomposición
es la siguiente.)*

**Caso 1 — fuera del corpus** ("costo mensual del parqueadero de motos"): el contexto NO contiene
tarifas de parqueadero.
| # | Afirmación | ¿Respaldada? |
|---|---|---|
| 1 | El parqueadero de motos cuesta $45.000/mes | **No** |
| 2 | Se exige carné estudiantil vigente | No (el carné aparece, pero no ligado al parqueadero) |
| 3 | El pago se hace en Tesorería | **No** |

**Faithfulness ≈ 0/3 = 0.0** → el modelo **alucinó**: inventó una cifra y un trámite ausentes del
contexto. La parte del prompt que *debía* evitarlo ("responde ÚNICAMENTE con el contexto … di que
no la tienes") es precisamente la que se incumple; un buen sistema debería **abstenerse**. Este es
el valor del caso: muestra que sin una instrucción de abstención estricta (y a veces aun con ella)
el LLM rellena huecos.

**Caso 2 — paráfrasis dentro del corpus** (p. ej. cancelación de período): las afirmaciones que
copian el contexto (hasta la décima semana, solicitud en el SIA, paz y salvo de biblioteca) están
**respaldadas**; una afirmación extra como "se reintegra automáticamente al siguiente período sin
trámite" **no** está en el contexto → baja la Faithfulness. 

**Análisis fuera del corpus:** la respuesta ideal es abstenerse. La instrucción clave del prompt es
la cláusula de "responde solo con el contexto y di explícitamente que no la tienes"; su ausencia o
debilidad es lo que permite la alucinación.

## SECCIÓN D — Declaración de uso de IA y reproducibilidad (5 pts)

**1. Herramienta(s) de IA usadas y 2 prompts clave.**
- Herramienta: asistente de IA (modelo de lenguaje) para redactar y depurar el código del notebook.
- Prompt clave 1: *"Implementa IoU y NMS greedy en Python puro y valida con assert que
  IoU((2,2,6,6),(4,4,8,8)) ≈ 0.143."*
- Prompt clave 2: *"Escribe la atención causal de una cabeza desde cero y verifica que coincide con
  F.scaled_dot_product_attention(..., is_causal=True)."*

**2. Proveedor, modelo de chat y modelo de embeddings efectivamente usados.**
- Proveedor de chat: **NVIDIA NIM** (`https://integrate.api.nvidia.com/v1`, SDK `openai`), con el
  modelo declarado en `MODELO_CHAT` (p. ej. `meta/llama-3.1-8b-instruct`; la celda de ping confirma
  el nombre exacto con `client.models.list()`). Si no hubiera clave, C.4 usaría el respaldo
  `faithfulness_cases.json`.
- Modelo de embeddings: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (oficial,
  según `embedding_info.json`), 384-d, normalizado. (Independiente del LLM de chat, regla R3.)

**3. Un caso en que la IA produjo algo incorrecto y cómo se corrigió (2 pts).**
En C.4, el caso fuera del corpus mostró que el LLM **alucinó** una tarifa de parqueadero inexistente
en el contexto. Se detectó al comparar cada afirmación contra el contexto (Faithfulness = 0) y se
corrige **endureciendo el prompt** (exigir abstención explícita) y verificando la fidelidad antes de
confiar en la respuesta. (Análogamente, un borrador inicial de la máscara causal usó `diagonal=0`,
que anulaba también la diagonal; el `assert` contra SDPA lo detectó y se corrigió a `diagonal=1`.)

In [ ]:
# [PROVISTO] 4. Reproducibilidad: semilla y versiones de librerías clave.
import importlib.metadata as md_
print("SEED =", SEED, "| D =", D)
for p in ["torch", "transformers", "sentence-transformers", "faiss-cpu", "openai", "rank-bm25", "tiktoken", "numpy", "pandas"]:
    try: print(f"  {p}: {md_.version(p)}")
    except Exception: print(f"  {p}: no instalado")
print("\nEl notebook corre de principio a fin sin errores (Reiniciar y ejecutar todo).")

## Revisión final

- El notebook corre de arriba hacia abajo sin errores.
- Todas las celdas `# [SU TURNO]` están resueltas (no quedan `raise NotImplementedError`).
- Las salidas (gráficas, tablas, asserts) están visibles.
- Ninguna celda ni salida muestra una API key (las claves se leen con `get_secret`).
- **Antes de entregar:** reemplace `STUDENT_NAME`, `STUDENT_ID` y `GROUP`, guarde como
  `Eval2_Apellido_Nombre.ipynb` y envíelo a `jipadillab@eafit.edu.co` con el asunto que imprime la
  celda de configuración, antes de las 11:00 a. m.

In [ ]:
# [PROVISTO] Verificación: ninguna API key en el notebook (ejecute tras guardar).
import re, json as _json
PATRONES = [r"nvapi-[A-Za-z0-9_\-]{10,}", r"gsk_[A-Za-z0-9]{10,}", r"hf_[A-Za-z0-9]{10,}"]
print("Asunto del correo:", ASUNTO)
print("Recuerde: no debe aparecer ninguna clave en celdas ni salidas.")